# GEMINI

In [1]:
import geopandas as gpd
import pandas as pd

lakes_clean = gpd.read_file(r'lakes_clean.gpkg')

In [3]:
import os
import tempfile
import numpy as np
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point, shape
from pysheds.grid import Grid
import rasterio.features
import odc.stac
import planetary_computer
import rioxarray


# Assume get_utm_epsg is defined

def get_utm_epsg(lon, lat):
    """Dynamically calculates the correct WGS84 UTM EPSG code."""
    utm_zone = int((lon + 180) / 6) + 1
    return 32600 + utm_zone

def calculate_watershed_area_cog(lake_row, catalog):
    """
    Streams a DEM via COG, delineates the upstream watershed with snapping, 
    and returns the total catchment area in km².
    """
    lake_geom = lake_row.geometry
    
    # A 0.30 degree buffer (~30km) is critical here to ensure you capture 
    # the entire watershed without truncating it at the edge of the bounding box.
    bbox = lake_geom.buffer(0.30).bounds 
    search = catalog.search(collections=["cop-dem-glo-30"], bbox=bbox)
    items = list(search.items())
    
    if not items:
        return {"status": "error", "watershed_area_km2": None, "reason": "No DEM data"}

    dem_dataset = odc.stac.load(
        items, bbox=bbox, crs="EPSG:4326", 
        resolution=0.0002777777777777778, patch_url=planetary_computer.sign
    )
    
    dem_array = dem_dataset["data"].squeeze()
    if 'longitude' in dem_array.dims:
        dem_array = dem_array.rename({'longitude': 'x', 'latitude': 'y'})
    dem_array = dem_array.rio.write_crs("EPSG:4326")
    
    # --- Find Pour Point ---
    if lake_geom.geom_type == 'Polygon':
        lon_pts, lat_pts = lake_geom.exterior.xy
    else:
        largest_poly = max(lake_geom.geoms, key=lambda p: p.area)
        lon_pts, lat_pts = largest_poly.exterior.xy
        
    lons_da = xr.DataArray(lon_pts, dims="points")
    lats_da = xr.DataArray(lat_pts, dims="points")
    elevations = dem_array.sel(x=lons_da, y=lats_da, method="nearest").values
    min_idx = np.nanargmin(elevations)
    pour_lon, pour_lat = lon_pts[min_idx], lat_pts[min_idx]
    
    # --- UTM Reprojection ---
    utm_crs = f"EPSG:{get_utm_epsg(pour_lon, pour_lat)}"
    dem_utm = dem_array.rio.reproject(utm_crs)
    pour_point_utm = gpd.GeoSeries([Point(pour_lon, pour_lat)], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
    
    # --- Pysheds Delineation with Snapping ---
    tmp = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
    tmp_path = tmp.name
    tmp.close()

    try:
        dem_utm.rio.to_raster(tmp_path)
        grid = Grid.from_raster(tmp_path)
        dem = grid.read_raster(tmp_path)

        pit_filled = grid.fill_pits(dem)
        flooded = grid.resolve_flats(pit_filled)
        fdir = grid.flowdir(flooded)
        
        acc = grid.accumulation(fdir)
        try:
            stream_mask = acc > 50 
            snapped_xy = grid.snap_to_mask(stream_mask, (pour_point_utm.x, pour_point_utm.y), return_dist=False)
            pour_x, pour_y = snapped_xy[0], snapped_xy[1]
        except Exception:
            pour_x, pour_y = pour_point_utm.x, pour_point_utm.y

        catchment_grid = grid.catchment(x=pour_x, y=pour_y, fdir=fdir, xytype='coordinate')

    finally:
        if os.path.exists(tmp_path):
            try: os.remove(tmp_path)
            except PermissionError: pass

    # --- Area Calculation ---
    catchment_array = np.asarray(catchment_grid)
    shapes = rasterio.features.shapes(
        catchment_array.astype(np.int16), mask=(catchment_array == 1), transform=grid.affine
    )
    catchment_polygons = [shape(geom) for geom, value in shapes if value == 1]
    
    if not catchment_polygons:
        return {"status": "error", "watershed_area_km2": None, "reason": "Delineation failed"}

    main_catchment = max(catchment_polygons, key=lambda a: a.area)
    catchment_gdf = gpd.GeoDataFrame(geometry=[main_catchment], crs=utm_crs)
    
    area_km2 = catchment_gdf.area.iloc[0] / 1_000_000

    return {
        "status": "success",
        "watershed_area_km2": round(area_km2, 3)
    }

In [3]:
import os
import time
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pystac_client import Client
import planetary_computer
import rioxarray

# Assume 'calculate_watershed_area_cog' and 'get_utm_epsg' are defined above

def process_watershed_areas(lakes_gdf, output_csv, max_retries=5, base_delay=5):
    """
    Processes lakes to calculate upstream watershed area (H7) in km2, 
    featuring automatic checkpointing and an exponential backoff retry loop.
    """
    # 1. Initialize STAC Client
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
    
    # 2. Load checkpoints to resume if the script stops
    processed_ids = set()
    if os.path.exists(output_csv):
        try:
            existing_df = pd.read_csv(output_csv)
            # Checking for GLAKE_ID explicitly
            if 'GLAKE_ID' in existing_df.columns:
                processed_ids = set(existing_df['GLAKE_ID'].dropna().astype(str))
            print(f"Resuming from checkpoint: {len(processed_ids)} lakes already processed.")
        except pd.errors.EmptyDataError:
            pass
            
    write_header = not os.path.exists(output_csv)
    
    # 3. Open output file in append mode
    with open(output_csv, 'a') as f:
        if write_header:
            f.write("GLAKE_ID,status,watershed_area_km2,error_reason\n")
            
        for index, lake_row in tqdm(lakes_gdf.iterrows(), total=len(lakes_gdf), desc="Calculating Watersheds"):
            # Extract GLAKE_ID (fallback to pandas index if missing)
            lake_id = str(lake_row.get('GLAKE_ID', index))
            
            # Skip if already processed
            if lake_id in processed_ids:
                continue
            
            # --- Exponential Backoff Retry Loop ---
            success = False
            for attempt in range(max_retries):
                try:
                    # Call the core COG pipeline function
                    result = calculate_watershed_area_cog(lake_row, catalog)
                    
                    status = result.get('status', 'unknown')
                    area = result.get('watershed_area_km2', '')
                    reason = result.get('reason', '')
                    
                    f.write(f"{lake_id},{status},{area},{reason}\n")
                    f.flush() 
                    success = True
                    break # Exit retry loop on success
                    
                except Exception as e:
                    error_msg = str(e)
                    # Handle API/Network timeouts
                    if any(x in error_msg for x in ["429", "502", "503", "timeout", "Connection"]):
                        delay = base_delay * (2 ** attempt)
                        tqdm.write(f"Network error on lake {lake_id}. Retrying in {delay}s...")
                        time.sleep(delay)
                    else:
                        # Structural or interpolation error, log and move on
                        f.write(f"{lake_id},error,,{error_msg.replace(',', ';')}\n")
                        f.flush()
                        success = True 
                        break
            
            if not success:
                f.write(f"{lake_id},error,,Max API retries exceeded\n")
                f.flush()

# --- Execution ---
# IMPORTANT: Pre-project to geographic coordinates to prevent bounding box errors
lakes_clean = lakes_clean.to_crs("EPSG:4326")
process_watershed_areas(lakes_clean, "hma_lake_watersheds_30m.csv")

Calculating Watersheds:  52%|█████▏    | 3606/6923 [8:10:46<7:31:26,  8.17s/it]


KeyboardInterrupt: 

In [4]:
# Faster watershed processing (same delineation logic) using controlled parallel I/O + checkpointing
import os
import time
import threading
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pystac_client import Client
import planetary_computer
import rioxarray


# Keep worker count conservative to avoid API throttling while still speeding up processing
MAX_WORKERS = 8
MAX_RETRIES = 5
BASE_DELAY = 5
OUT_CSV = r"D:\Rubel\M.Tech\MTP\Phase 2\hma_lake_watersheds_30m copy.csv"

# Ensure geographic CRS for bbox search
lakes_for_run = lakes_clean.to_crs("EPSG:4326").copy()

# Auto-detect ID column
if "GLAKE_ID" in lakes_for_run.columns:
    id_col = "GLAKE_ID"
elif "glake_id" in lakes_for_run.columns:
    id_col = "glake_id"
else:
    id_col = None

thread_local = threading.local()


def get_catalog():
    """One STAC client per worker thread to avoid cross-thread client reuse issues."""
    if not hasattr(thread_local, "catalog"):
        thread_local.catalog = Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=planetary_computer.sign_inplace,
        )
    return thread_local.catalog


def process_one(index_row):
    index, lake_row = index_row
    lake_id = str(lake_row[id_col]) if id_col else str(index)

    for attempt in range(MAX_RETRIES):
        try:
            catalog = get_catalog()
            result = calculate_watershed_area_cog(lake_row, catalog)
            return {
                "GLAKE_ID": lake_id,
                "status": result.get("status", "unknown"),
                "watershed_area_km2": result.get("watershed_area_km2", None),
                "error_reason": result.get("reason", ""),
            }
        except Exception as e:
            msg = str(e)
            is_transient = any(x in msg for x in ["429", "502", "503", "timeout", "Connection"])
            if is_transient and attempt < (MAX_RETRIES - 1):
                time.sleep(BASE_DELAY * (2 ** attempt))
                continue
            return {
                "GLAKE_ID": lake_id,
                "status": "error",
                "watershed_area_km2": None,
                "error_reason": msg,
            }


# Resume support
processed_ids = set()
if os.path.exists(OUT_CSV):
    try:
        old = pd.read_csv(OUT_CSV)
        if "GLAKE_ID" in old.columns:
            processed_ids = set(old["GLAKE_ID"].dropna().astype(str))
            print(f"Resuming from checkpoint: {len(processed_ids)} already done")
    except Exception:
        pass

# Build task list excluding already processed
tasks = []
for idx, row in lakes_for_run.iterrows():
    lake_id = str(row[id_col]) if id_col else str(idx)
    if lake_id not in processed_ids:
        tasks.append((idx, row))

print(f"Total lakes: {len(lakes_for_run)}")
print(f"Pending lakes: {len(tasks)}")
print(f"Workers: {MAX_WORKERS}")

if len(tasks) == 0:
    print("Nothing to process. Output is up to date.")
else:
    write_header = not os.path.exists(OUT_CSV)
    with open(OUT_CSV, "a", encoding="utf-8") as f:
        if write_header:
            f.write("GLAKE_ID,status,watershed_area_km2,error_reason\n")

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process_one, task) for task in tasks]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Watersheds", unit="lake"):
                row = fut.result()
                err = str(row["error_reason"]).replace(",", ";").replace("\n", " ")
                f.write(f"{row['GLAKE_ID']},{row['status']},{row['watershed_area_km2']},{err}\n")
                f.flush()

    print(f"Saved: {OUT_CSV}")
    out_df = pd.read_csv(OUT_CSV)
    print(out_df[["GLAKE_ID", "watershed_area_km2"]].head())

Resuming from checkpoint: 3538 already done
Total lakes: 6923
Pending lakes: 3348
Workers: 8


Watersheds:   0%|          | 0/3348 [00:00<?, ?lake/s]

d:\anaconda\geo_dl\Lib\site-packages\rasterio\warp.py:385: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


Saved: D:\Rubel\M.Tech\MTP\Phase 2\hma_lake_watersheds_30m copy.csv
          GLAKE_ID  watershed_area_km2
0  GL077099E35395N               0.041
1  GL078223E34278N               0.047
2  GL078079E34401N               0.048
3  GL078069E34414N               0.093
4  GL078100E34520N               0.096


In [5]:
lakes_clean.shape

(6923, 27)

In [6]:
out_df.shape

(6915, 4)

In [13]:
# Append watershed area to lakes_clean by matching lake ID
import os
import numpy as np
import pandas as pd

# Pick watershed CSV path
watershed_csv = OUT_CSV if "OUT_CSV" in globals() else r"D:\Rubel\M.Tech\MTP\Phase 2\hma_lake_watersheds_30m copy.csv"
if not os.path.exists(watershed_csv):
    raise FileNotFoundError(f"Watershed CSV not found: {watershed_csv}")

# Detect ID column in lakes_clean
if "GLAKE_ID" in lakes_clean.columns:
    lakes_id_col = "GLAKE_ID"
elif "glake_id" in lakes_clean.columns:
    lakes_id_col = "glake_id"
else:
    raise KeyError("No ID column in lakes_clean. Expected 'GLAKE_ID' or 'glake_id'.")

# Read watershed results CSV
ws = pd.read_csv(watershed_csv)

# Detect ID column in CSV
if "GLAKE_ID" in ws.columns:
    ws_id_col = "GLAKE_ID"
elif "glake_id" in ws.columns:
    ws_id_col = "glake_id"
else:
    raise KeyError("No ID column in watershed CSV. Expected 'GLAKE_ID' or 'glake_id'.")

if "watershed_area_km2" not in ws.columns:
    raise KeyError("Column 'watershed_area_km2' not found in watershed CSV.")

# Keep last non-null watershed value per ID (handles retries/append duplicates)
ws2 = ws.copy()
ws2["_id_norm"] = ws2[ws_id_col].astype(str).str.strip()
ws2["watershed_area_km2"] = pd.to_numeric(ws2["watershed_area_km2"], errors="coerce")
ws2 = ws2.sort_values("watershed_area_km2", na_position="first")
ws_latest = ws2.drop_duplicates("_id_norm", keep="last")[["_id_norm", "watershed_area_km2"]]

# Merge into lakes_clean
lakes_clean["_id_norm"] = lakes_clean[lakes_id_col].astype(str).str.strip()
lakes_clean = lakes_clean.merge(ws_latest, on="_id_norm", how="left", suffixes=("", "_from_csv"))

# If watershed_area_km2 already exists, fill only missing values; otherwise create it
if "watershed_area_km2" in lakes_clean.columns and "watershed_area_km2_from_csv" in lakes_clean.columns:
    lakes_clean["watershed_area_km2"] = lakes_clean["watershed_area_km2"].combine_first(lakes_clean["watershed_area_km2_from_csv"])
    lakes_clean = lakes_clean.drop(columns=["watershed_area_km2_from_csv"])

# Cleanup helper column
lakes_clean = lakes_clean.drop(columns=["_id_norm"])

# Quick check
matched = lakes_clean["watershed_area_km2"].notna().sum() if "watershed_area_km2" in lakes_clean.columns else 0
print(f"lakes_clean rows: {len(lakes_clean)}")
print(f"Rows with watershed_area_km2: {matched}")
print(lakes_clean[[lakes_id_col, "watershed_area_km2"]].head())

lakes_clean rows: 6923
Rows with watershed_area_km2: 6923
          GLAKE_ID  watershed_area_km2
0  GL077099E35395N               0.041
1  GL078223E34278N               0.047
2  GL078079E34401N               0.048
3  GL078069E34414N               0.093
4  GL078100E34520N               0.096


In [15]:
lakes_clean.to_file(r'lakes_clean.gpkg',driver = 'GPKG')